# 10. Bias in the data: heart-failure death

Same `ebdai` workflow as the Titanic demo, on the heart-failure table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025). The sensitive attribute is `sex` (0 = female, 1 = male); the label is `death`.

On [Google Colab](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/10_bias_heart_failure.ipynb), run the setup cell below first. It follows the workshop install guide: clone this repository, move into `EBD-AI`, and `pip install` it. The workshop table ships with that install. If the next cell cannot import `ebdai` just after the install, restart the runtime and run every cell again.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HAISymbiosis/EBD-AI/blob/main/Demos/10_bias_heart_failure.ipynb)


In [ ]:
!git clone -q https://github.com/HAISymbiosis/EBD-AI.git
%cd EBD-AI
!pip install -q .


In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_heart_failure, outcome_rates_by_group,
    plot_outcome_rates, plot_winning_rules_by_group, fairness_report,
    parse_printed_rules, winning_rules_by_group,
)

frame, sensitive = load_heart_failure()
X, y = features_and_target(frame, 'death')
print(X.dtypes)
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Death rate by sex')

age                         float64
anaemia                      object
creatinine_phosphokinase      int64
diabetes                     object
ejection_fraction             int64
high_blood_pressure          object
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                          object
smoking                      object
dtype: object
   group    n  n_positive  positive_rate
0      0  105          34       0.323810
1      1  194          62       0.319588


<Axes: title={'center': 'Death rate by sex'}, xlabel='Group', ylabel='Positive rate'>

Fit a short-budget fuzzy classifier and inspect group-wise rule firings.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=6, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))
counts = winning_rules_by_group(
    clf, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report or ''),
)
plot_winning_rules_by_group(counts, title='Winning heart-failure rules by sex')

------------
ACCURACY
Train performance: 0.795
Test performance: 0.6060606060606061
------------
MATTHEW CORRCOEF
Train performance: 0.4162463467383999
Test performance: 0.17534845441898042
------------
Rules for consequent: 0
----------------
IF serum_creatinine IS Low WITH DS 0.5408260303382613, ACC 0.7932960893854749, WGHT 1.0

Rules for consequent: 1
----------------
IF ejection_fraction IS Low WITH DS 0.06204445858071811, ACC 0.8181818181818182, WGHT 1.0
IF anaemia IS 0 AND serum_sodium IS Medium AND sex IS 0 WITH DS 0.006113775659428545, ACC 1.0, WGHT 1.0
IF creatinine_phosphokinase IS Low AND platelets IS Medium WITH DS 0.0215734818868575, ACC 0.7142857142857143, WGHT 1.0


   group   n  selection_rate       tpr   fpr       fnr   tnr
0      1  67        0.029851  0.066667  0.00  0.933333  1.00
1      0  32        0.093750  0.166667  0.05  0.833333  0.95
demographic_parity_difference    0.063899
demographic_parity_ratio         0.318408
equalized_odds_difference        0.100000
e

<Axes: title={'center': 'Winning heart-failure rules by sex'}, xlabel='Winning rule', ylabel='Share of group'>